# γ-CKLSearch — Chumbalov et al. 2024 (UAI)

Implementation of Algorithm 3 (γ-CKLSearch) from *Fast Interactive Search under a Scale-Free Comparison Oracle*.

Key differences from GAUSSSEARCH (2020):
- **Oracle model**: γ-CKL instead of Probit. p(pick i) = ‖x_j - x_t‖^γ / (‖x_i - x_t‖^γ + ‖x_j - x_t‖^γ)
- **Belief representation**: full posterior over n items instead of Gaussian (μ, Σ) in ℝ^d
- **Query selection**: same mirror-descent shape, but scoring uses posterior directly
- **γ parameter**: controls oracle discriminating power independent of dimension d

Reference values from the paper (user study): D=5, γ=3, r=2, σ_ε=0.1

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict

rng = np.random.default_rng(seed=42)

## Dataset

In [2]:
def make_dataset(n, d, rng):
    """Generate n items uniformly in [-1, 1]^d."""
    return rng.uniform(-1, 1, size=(n, d))


X = make_dataset(n=100, d=5, rng=rng)
print(f"X shape: {X.shape}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")

X shape: (100, 5)
X range: [-0.989, 0.998]


## γ-CKL oracle model

$$p_{x_i, x_j, x_t} = \frac{\|x_j - x_t\|^\gamma}{\|x_i - x_t\|^\gamma + \|x_j - x_t\|^\gamma}$$

γ → ∞ makes the oracle nearly deterministic (picks closest). γ → 0 makes it 1/2 (random). Scale-free: multiplying all embeddings by a constant does not change the probability.

In [3]:
def gamma_ckl_prob(x_i, x_j, x_t, gamma):
    """P(oracle picks i | query (i,j), target t)."""
    d_i = np.linalg.norm(x_i - x_t)
    d_j = np.linalg.norm(x_j - x_t)
    if d_i == 0:
        return 1.0
    if d_j == 0:
        return 0.0
    return (d_j ** gamma) / (d_i ** gamma + d_j ** gamma)


def query_oracle(x_i, x_j, x_t, gamma, rng):
    """Sample an oracle answer. Returns 0 (chose i) or 1 (chose j)."""
    p_i = gamma_ckl_prob(x_i, x_j, x_t, gamma)
    return 0 if rng.random() < p_i else 1


# smoke test: for high γ, oracle almost always picks the closer item
X_test = make_dataset(n=3, d=2, rng=rng)
x_i, x_j, x_t = X_test[0], X_test[1], X_test[2]
p = gamma_ckl_prob(x_i, x_j, x_t, gamma=5)
print(f"distances: i={np.linalg.norm(x_i-x_t):.3f}, j={np.linalg.norm(x_j-x_t):.3f}")
print(f"P(pick i) = {p:.3f}")
answers = [query_oracle(x_i, x_j, x_t, gamma=5, rng=rng) for _ in range(1000)]
print(f"empirical P(pick i) = {answers.count(0)/1000:.3f}")

distances: i=0.466, j=0.997
P(pick i) = 0.978
empirical P(pick i) = 0.969


## Query selection (mirror-descent heuristic)

Algorithm 3 line 9 in the paper:

$$i = \arg\min_{i \notin U} p_i^m \cdot \|x_i - \tilde z_1\|^2$$

Two interpretations:

- **`"paper_literal"`** — score = `P * d²` as written. Note: this actually favors items with SMALL P.
- **`"paper_textual"`** — score = `d² / P`. The text says "favors near AND more probable points" — this matches that description.

We expose the choice as `score_mode` so we can compare.

In [4]:
def sample_mirror(X, P, used, r=2.0, score_mode="paper_textual", eps=1e-12):
    """Pick next query (i, j) via the SAMPLEMIRROR heuristic (Algorithm 3, lines 5-9).

    Steps:
      1. Compute posterior mean μ̄ and covariance Σ̄ from P.
      2. Find top eigenvector v_max of Σ̄ and its eigenvalue λ_max.
      3. Construct proto-queries z̃1, z̃2 = μ̄ ± r√λ_max v_max.
      4. Pick items i, j closest to z̃1, z̃2 under the scoring rule.

    score_mode:
      "paper_literal"  score = P_i * ||x_i - z̃||²   (arg min from Algorithm 3 line 9)
      "paper_textual"  score = ||x_i - z̃||² / P_i   (matches "favors near and probable")
    """
    n, d = X.shape

    mu = (P[:, None] * X).sum(axis=0)
    diff = X - mu
    Sigma = (P[:, None, None] * diff[:, :, None] * diff[:, None, :]).sum(axis=0)

    eigvals, eigvecs = np.linalg.eigh(Sigma)
    lam_max = eigvals[-1]
    v_max = eigvecs[:, -1]

    z1 = mu + r * np.sqrt(max(lam_max, eps)) * v_max
    z2 = mu - r * np.sqrt(max(lam_max, eps)) * v_max

    def score(z):
        d2 = np.sum((X - z) ** 2, axis=1)
        if score_mode == "paper_literal":
            return P * d2
        elif score_mode == "paper_textual":
            return d2 / (P + eps)
        else:
            raise ValueError(f"unknown score_mode: {score_mode}")

    def pick(z, excl):
        s = score(z).copy()
        for u in excl:
            s[u] = np.inf
        return int(np.argmin(s))

    i = pick(z1, used)
    j = pick(z2, used | {i})
    return i, j

## Posterior update (Bayes)

After observing y ∈ {i, j}:

$$p_k^{m+1} = \frac{p_k^m \cdot p_{x_i, x_j, x_k}(y)}{Z}$$

Vectorised over all n candidate targets.

In [5]:
def bayes_update(X, P, i, j, y, gamma, eps=1e-12):
    """Update posterior P after observing oracle answer y (y == i or y == j)."""
    d_i = np.linalg.norm(X - X[i], axis=1)
    d_j = np.linalg.norm(X - X[j], axis=1)

    d_i_g = d_i ** gamma
    d_j_g = d_j ** gamma
    denom = d_i_g + d_j_g + eps

    if y == i:
        likelihood = d_j_g / denom
    else:
        likelihood = d_i_g / denom

    P_new = P * likelihood
    Z = P_new.sum()
    if Z < eps:
        return P.copy()
    return P_new / Z

## Main search loop (Algorithm 3)

Stopping criterion: `target ∈ {i, j}`, matching the paper. This is the "in_query" stopping mode we already use for GAUSSSEARCH.

In [6]:
def gamma_ckl_search(X, target_idx, gamma, r=2.0, max_queries=200,
                     rng=None, score_mode="paper_textual"):
    """Run γ-CKLSearch until target appears in a query pair or budget exhausted.

    Returns:
      dict with keys:
        found       : bool — did the target appear in a query?
        n_queries   : int — number of queries asked
        history     : dict with P_target, queries, argmax per step
    """
    if rng is None:
        rng = np.random.default_rng()
    n = len(X)
    P = np.ones(n) / n
    used = set()
    x_t = X[target_idx]

    history = {"P_target": [], "queries": [], "argmax": []}

    for step in range(max_queries):
        i, j = sample_mirror(X, P, used, r=r, score_mode=score_mode)
        used |= {i, j}

        y_idx = query_oracle(X[i], X[j], x_t, gamma, rng)
        y = i if y_idx == 0 else j

        history["P_target"].append(P[target_idx])
        history["queries"].append((i, j))
        history["argmax"].append(int(np.argmax(P)))

        if target_idx in (i, j):
            return {"found": True, "n_queries": step + 1, "history": history}

        P = bayes_update(X, P, i, j, y, gamma)

    return {"found": False, "n_queries": max_queries, "history": history}

## Smoke test — single search

Trajectory of P[target] should climb toward 1 as evidence accumulates.

In [7]:
rng = np.random.default_rng(seed=0)
X = make_dataset(n=100, d=5, rng=rng)
target_idx = 42

for mode in ["paper_textual", "paper_literal"]:
    result = gamma_ckl_search(
        X, target_idx, gamma=5, r=2.0, max_queries=50,
        rng=np.random.default_rng(seed=0), score_mode=mode,
    )
    print(f"[{mode}] found={result['found']}, queries={result['n_queries']}")
    print(f"  P[target] first 10 steps:")
    for step, p in enumerate(result['history']['P_target'][:10]):
        argmax = result['history']['argmax'][step]
        marker = "  <-- target is argmax" if argmax == target_idx else ""
        print(f"    step {step:2d}: P[target] = {p:.4f}, argmax = {argmax}{marker}")
    print()

[paper_textual] found=True, queries=3
  P[target] first 10 steps:
    step  0: P[target] = 0.0100, argmax = 0
    step  1: P[target] = 0.0322, argmax = 23
    step  2: P[target] = 0.0620, argmax = 23

[paper_literal] found=True, queries=50
  P[target] first 10 steps:
    step  0: P[target] = 0.0100, argmax = 0
    step  1: P[target] = 0.0322, argmax = 23
    step  2: P[target] = 0.0364, argmax = 73
    step  3: P[target] = 0.0441, argmax = 73
    step  4: P[target] = 0.0460, argmax = 73
    step  5: P[target] = 0.0475, argmax = 73
    step  6: P[target] = 0.0533, argmax = 73
    step  7: P[target] = 0.0673, argmax = 42  <-- target is argmax
    step  8: P[target] = 0.0647, argmax = 73
    step  9: P[target] = 0.0776, argmax = 42  <-- target is argmax



## Batch eval — 20 targets

Compare the two scoring modes head-to-head on the same targets with the same RNG.

In [8]:
N_TARGETS = 20
n, d = 100, 5
gamma_val = 5

X_setup = make_dataset(n=n, d=d, rng=np.random.default_rng(seed=0))

results = defaultdict(list)
for mode in ["paper_textual", "paper_literal"]:
    for target_idx in range(N_TARGETS):
        result = gamma_ckl_search(
            X_setup, target_idx, gamma=gamma_val, r=2.0, max_queries=100,
            rng=np.random.default_rng(seed=1000 + target_idx),
            score_mode=mode,
        )
        results[mode].append(result["n_queries"])

print(f"{'mode':<20} | {'mean':>6} {'med':>4} {'max':>4}")
print("-" * 40)
for mode in ["paper_textual", "paper_literal"]:
    r_arr = np.array(results[mode])
    print(f"{mode:<20} | {r_arr.mean():>6.2f} {np.median(r_arr):>4.1f} {r_arr.max():>4d}")

mode                 |   mean  med  max
----------------------------------------
paper_textual        |   7.50  7.0   13
paper_literal        |  47.00 50.0   50


## Scaling experiment — n ∈ {50, 100, 500, 1000}

Matches the same shape as `01_gauss_search.ipynb` cell 10-13 so we can drop the numbers next to GAUSSSEARCH's for direct comparison.

In [9]:
def run_scaling_gamma_ckl(n_values, d=5, gamma=5, r=2.0, n_trials=200,
                            seed=0, score_mode="paper_textual"):
    """Run γ-CKLSearch on n_trials random targets for each n. Return dict n -> list of query counts."""
    out = defaultdict(list)
    for n in n_values:
        rng = np.random.default_rng(seed=seed + n)
        X = make_dataset(n=n, d=d, rng=rng)
        for trial in range(n_trials):
            target_idx = rng.integers(0, n)
            res = gamma_ckl_search(
                X, target_idx, gamma=gamma, r=r,
                max_queries=200, rng=rng, score_mode=score_mode,
            )
            out[n].append(res["n_queries"])
    return out


n_values = [50, 100, 500, 1000]
scaling_gamma_ckl = run_scaling_gamma_ckl(n_values, d=5, gamma=5, n_trials=100, seed=0)

print(f"{'n':>6} | {'mean':>6} {'med':>5} {'p75':>5} {'p90':>5} {'p95':>5} {'max':>5}")
print("-" * 50)
for n in n_values:
    arr = np.array(scaling_gamma_ckl[n])
    print(f"{n:>6} | {arr.mean():>6.2f} {np.median(arr):>5.1f} "
          f"{np.percentile(arr, 75):>5.1f} {np.percentile(arr, 90):>5.1f} "
          f"{np.percentile(arr, 95):>5.1f} {arr.max():>5d}")

     n |   mean   med   p75   p90   p95   max
--------------------------------------------------
    50 |   5.00   5.0   6.0   7.1   8.0    10
   100 |   6.28   6.0   8.0   9.0  10.0    14
   500 |  11.37  10.0  13.0  18.0  21.0    29
  1000 |  13.76  12.0  15.2  22.1  27.0    47


## Head-to-head vs GAUSSSEARCH

Load the 2020 implementation from `01_gauss_search.ipynb` (copy-paste `gauss_search` and its helpers into a cell above, or import them). Run both on the same n_values, same seed, same targets. Compare mean queries.

For the scenery dataset later: load the E_work embeddings from `scenery-search/data/scenery_embedding.npz`, treat it as X, and run both algorithms on the 100-target eval set from `05_closed_loop_v2.ipynb`.

In [10]:
# Placeholder for the head-to-head comparison.
# Copy the `gauss_search` function (and helpers: bisecting_hyperplane, sample_mirror,
# adf_update) from 01_gauss_search.ipynb into a cell above, then run:
#
#   for n in n_values:
#       X = make_dataset(n=n, d=5, rng=np.random.default_rng(seed=0))
#       for trial in range(100):
#           target = ...
#           gs_res = gauss_search(X, target, sigma_eps=0.20, rng=...)
#           gc_res = gamma_ckl_search(X, target, gamma=5, r=2.0, rng=...)
#           # collect gs_res["n_queries"] vs gc_res["n_queries"]
#
# For the scenery-search comparison on real embeddings, do this in a separate
# notebook under scenery-search/notebooks/ where the E_work loading code already lives.
pass

## Next steps

1. Implement head-to-head on synthetic data (n=100, d=5) with 100 trials — expected: γ-CKL should be competitive or better if the oracle model matches the data-generating process.
2. Port to scenery data: replace `make_dataset` output with `E_work / X_scale` from `scenery_embedding.npz`.
3. On scenery: run against the same 100-target eval set used in `05_closed_loop_v2.ipynb` so numbers are directly comparable.
4. Tune (γ, r) with a small grid. Paper defaults: γ=3, r=2 for D=5. Try (3, 2), (5, 2), (5, 3).
5. If γ-CKL wins consistently, propose to Sir integrating it into the app as an alternative search engine.